# 01 Real Smoke Test

            Local-first real smoke test. This runs Qwen2.5-1.5B on 10 real data samples for each required condition and writes about 60 JSONL records.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import math
import random
import re
import string
import time

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_SIZE = "1.5B"
SMOKE_CONDITIONS = [
    "reasoning_direct",
    "reasoning_cot",
    "context_none",
    "context_oracle",
    "knowledge_closed",
    "knowledge_oracle",
]
MAX_SAMPLES_PER_CONDITION = 10
SHARD_ID = 0
NUM_SHARDS = 1
SEED = 42
TEMPERATURE = 0.0
TOP_P = 1.0
MAX_NEW_TOKENS = 384
BACKEND = "transformers"
LOCAL_FILES_ONLY = False
REQUIRE_REAL_DATA = True

DATA_DIR = Path("slm_limits_data")
OUTPUT_DIR = Path("outputs")
MODEL_TAG = MODEL_SIZE.replace(".", "p")
PROMPT_TEMPLATE = "smoke_v4"
OUTPUT_PATH = OUTPUT_DIR / f"records_smoke_qwen2p5_{MODEL_TAG}_real_six_conditions_v4_shard{SHARD_ID}.jsonl"
OVERWRITE_OUTPUT = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)

In [ ]:
CONDITION_TO_DATA = {
    "reasoning_direct": DATA_DIR / "reasoning_gsm8k.jsonl",
    "reasoning_cot": DATA_DIR / "reasoning_gsm8k.jsonl",
    "context_none": DATA_DIR / "context_hotpotqa.jsonl",
    "context_oracle": DATA_DIR / "context_hotpotqa.jsonl",
    "knowledge_closed": DATA_DIR / "knowledge_data.jsonl",
    "knowledge_oracle": DATA_DIR / "knowledge_data.jsonl",
}


def read_jsonl(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing data file: {path}. Run 00-prepare-data.ipynb first.")
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def validate_real_rows(path, rows):
    if len(rows) < MAX_SAMPLES_PER_CONDITION:
        raise ValueError(f"{path} has only {len(rows)} rows; need at least {MAX_SAMPLES_PER_CONDITION}.")
    if not REQUIRE_REAL_DATA:
        return
    fallback_count = sum(1 for row in rows if row.get("metadata", {}).get("source") == "local_fallback")
    if fallback_count == len(rows):
        raise ValueError(
            f"{path} looks like local fallback/toy data. Restore the real Kaggle/HF JSONL files before running real smoke."
        )


def append_jsonl(path, row):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def successful_keys(path):
    if OVERWRITE_OUTPUT and path.exists():
        path.unlink()
        return set()
    keys = set()
    if not path.exists():
        return keys
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            if row.get("status") == "success":
                keys.add(row.get("unique_key"))
    return keys


def unique_key(sample, condition):
    parts = [MODEL_NAME, sample["axis"], sample["dataset"], condition, sample["sample_id"], PROMPT_TEMPLATE]
    return "::".join(str(part) for part in parts)

In [ ]:
ARTICLES = {"a", "an", "the"}


def normalize_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = [token for token in text.split() if token not in ARTICLES]
    return " ".join(tokens)


def token_f1(prediction, gold):
    pred_tokens = normalize_text(prediction).split()
    gold_tokens = normalize_text(gold).split()
    if not pred_tokens and not gold_tokens:
        return 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0
    common = {}
    for token in pred_tokens:
        common[token] = min(pred_tokens.count(token), gold_tokens.count(token))
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def extract_after_final_answer(text):
    match = re.search(r"final answer\s*:\s*(.+)", str(text), flags=re.IGNORECASE | re.DOTALL)
    if match:
        text = match.group(1)
    lines = [line.strip() for line in str(text).splitlines() if line.strip()]
    return lines[-1] if lines else ""


def extract_short_qa_answer(text):
    match = re.search(r"final answer\s*:\s*(.*)", str(text), flags=re.IGNORECASE | re.DOTALL)
    if match:
        text = match.group(1)
    lines = [line.strip() for line in str(text).splitlines() if line.strip()]
    if not lines:
        return ""
    answer = re.sub(r"^(the answer is|answer is|answer)\s*:?\s*", "", lines[0], flags=re.IGNORECASE)
    return answer.strip(" .")


def has_final_answer(text):
    return re.search(r"final answer\s*:", str(text), flags=re.IGNORECASE) is not None


def extract_json_answer(text):
    try:
        parsed = json.loads(str(text).strip())
    except json.JSONDecodeError:
        return ""
    return str(parsed.get("answer", "")) if isinstance(parsed, dict) else ""


def extract_numeric(text):
    numbers = re.findall(r"-?\d+(?:\.\d+)?", str(text).replace(",", ""))
    if not numbers:
        return ""
    value = numbers[-1]
    return value[:-2] if value.endswith(".0") else value


def extract_answer(raw_text, axis, condition):
    if "json" in condition:
        parsed = extract_json_answer(raw_text)
        if parsed:
            return parsed
    if axis == "reasoning":
        text = extract_after_final_answer(raw_text) if has_final_answer(raw_text) else raw_text
        numeric = extract_numeric(text)
        return numeric or extract_after_final_answer(raw_text)
    return extract_short_qa_answer(raw_text)


def score_prediction(prediction, gold, axis):
    if axis == "reasoning":
        pred_num = extract_numeric(prediction)
        gold_num = extract_numeric(gold)
        accuracy = float(pred_num != "" and pred_num == gold_num)
        exact_match = accuracy
        f1 = accuracy
    else:
        exact_match = float(normalize_text(prediction) == normalize_text(gold))
        f1 = token_f1(prediction, gold)
        accuracy = exact_match
    return exact_match, f1, accuracy

In [ ]:
def build_prompt(sample, condition):
    question = sample["question"]
    context = sample.get("context", "")
    if condition == "reasoning_direct":
        return f"Solve the math problem. Return only one number. Do not explain.\n\nProblem: {question}\n\nFinal answer:"
    if condition == "reasoning_cot":
        return f"Solve the math problem step by step. End with \"Final answer: <number>\".\n\nProblem: {question}\n\nFinal answer:"
    if condition == "context_none":
        return f"Answer with a short phrase only. Do not explain.\n\nQuestion: {question}\n\nFinal answer:"
    if condition == "context_oracle":
        return f"Answer with a short phrase only. Do not explain. Use the context when possible.\n\nContext:\n{context}\n\nQuestion: {question}\n\nFinal answer:"
    if condition == "knowledge_closed":
        return f"Answer with a short phrase only. Do not explain.\n\nQuestion: {question}\n\nFinal answer:"
    if condition == "knowledge_oracle":
        return f"Answer with a short phrase only. Do not explain. Use the context when possible.\n\nContext:\n{context}\n\nQuestion: {question}\n\nFinal answer:"
    raise ValueError(f"Unknown condition: {condition}")


def mock_generate(sample, condition):
    gold = sample["gold_answer"]
    if condition.endswith("_cot"):
        return f"We solve it carefully using the quantities in the problem.\nFinal answer: {gold}"
    if condition.endswith("_none") or condition.endswith("_closed"):
        # Keep local smoke deterministic but imperfect so summaries show meaningful deltas.
        if int(re.findall(r"\d+", sample["sample_id"])[-1]) % 4 == 0:
            return "Final answer: unknown"
    return f"Final answer: {gold}"


def load_transformers_model():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=LOCAL_FILES_ONLY)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=dtype,
        device_map="auto",
        local_files_only=LOCAL_FILES_ONLY,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer, model


def transformers_generate(tokenizer, model, prompt):
    import torch

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    start = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = time.perf_counter() - start
    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    input_tokens = int(inputs["input_ids"].shape[-1])
    output_tokens = int(generated_ids.shape[-1])
    memory_gb = 0.0
    if torch.cuda.is_available():
        memory_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
    return text, latency, input_tokens, output_tokens, memory_gb

In [ ]:
def select_samples(condition):
    rows = read_jsonl(CONDITION_TO_DATA[condition])
    validate_real_rows(CONDITION_TO_DATA[condition], rows)
    rows = [row for index, row in enumerate(rows) if index % NUM_SHARDS == SHARD_ID]
    return rows[:MAX_SAMPLES_PER_CONDITION]


completed = successful_keys(OUTPUT_PATH)
samples_by_condition = {condition: select_samples(condition) for condition in SMOKE_CONDITIONS}
condition_counts = {condition: len(samples) for condition, samples in samples_by_condition.items()}
print({"validated_conditions": condition_counts, "model_name": MODEL_NAME, "backend": BACKEND})

model_bundle = load_transformers_model() if BACKEND == "transformers" else None
records_written = 0

for condition in SMOKE_CONDITIONS:
    samples = samples_by_condition[condition]
    for sample in samples:
        key = unique_key(sample, condition)
        if key in completed:
            continue

        prompt = build_prompt(sample, condition)
        start = time.perf_counter()
        status = "success"
        error_message = ""
        input_tokens = len(prompt.split())
        output_tokens = 0
        max_memory_gb = 0.0

        try:
            if BACKEND == "transformers":
                tokenizer, model = model_bundle
                raw, latency, input_tokens, output_tokens, max_memory_gb = transformers_generate(tokenizer, model, prompt)
            else:
                raw = mock_generate(sample, condition)
                latency = time.perf_counter() - start
                output_tokens = len(raw.split())

            prediction = extract_answer(raw, sample["axis"], condition)
            exact_match, f1, accuracy = score_prediction(prediction, sample["gold_answer"], sample["axis"])
            tokens_per_second = output_tokens / latency if latency > 0 else 0.0
        except Exception as exc:
            raw = ""
            prediction = ""
            exact_match = f1 = accuracy = 0.0
            latency = time.perf_counter() - start
            tokens_per_second = 0.0
            status = "error"
            error_message = repr(exc)

        record = {
            "unique_key": key,
            "sample_id": sample["sample_id"],
            "axis": sample["axis"],
            "dataset": sample["dataset"],
            "condition": condition,
            "model_name": MODEL_NAME,
            "model_size": MODEL_SIZE,
            "prompt_template": PROMPT_TEMPLATE,
            "question": sample["question"],
            "context": sample.get("context", ""),
            "gold_answer": sample["gold_answer"],
            "prediction_raw": raw,
            "prediction_normalized": prediction,
            "exact_match": exact_match,
            "f1": f1,
            "accuracy": accuracy,
            "additional_metrics": {},
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "latency_seconds": latency,
            "tokens_per_second": tokens_per_second,
            "max_memory_allocated_gb": max_memory_gb,
            "context_truncated": False,
            "status": status,
            "error_message": error_message,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        }
        append_jsonl(OUTPUT_PATH, record)
        completed.add(key)
        records_written += 1

print({"output_path": str(OUTPUT_PATH), "records_written": records_written, "condition_counts": condition_counts})